In [16]:
# 1.导入相关包
import os
import dotenv
from langchain_classic.memory import ConversationTokenBufferMemory, ConversationSummaryMemory, \
    ConversationSummaryBufferMemory
from langchain_core.messages import ChatMessage
from langchain_openai import ChatOpenAI

# 2.创建大模型
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("DEEPSEEK_BASE_URL")


# 1、 ConversationTokenBufferMemory的使用
举例1：

In [3]:
###deepseek-chat 是深度求索的模型，虽然它可以通过 ChatOpenAI 类进行调用（兼容 OpenAI 的 API 格式），但它并没有实现 OpenAI 官方模型对应的 get_num_tokens_from_messages() 方法（该方法最初是为 OpenAI 官方模型设计的），无法完成对话消息的 token 统计，导致 ConversationTokenBufferMemory 无法正常执行 save_context() 操作


llm = ChatOpenAI(model = "deepseek-chat")

# 3.定义ConversationTokenBufferMemory对象
memory = ConversationTokenBufferMemory(
    llm = llm,
    max_token_limit=10 #设置token上限，默认值为2000
)

#添加对话
memory.save_context({"input":"你好吗？"},{"output":"我很好，谢谢！"})
memory.save_context({"input":"今天天气如何？"},{"output":"晴天，25℃"})

#查看当前记忆
print(memory.load_memory_variables({}))

C:\Users\Song\AppData\Local\Temp\ipykernel_31464\659814362.py:15: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationTokenBufferMemory(


NotImplementedError: get_num_tokens_from_messages() is not presently implemented for model deepseek-chat. See https://platform.openai.com/docs/guides/text-generation/managing-tokens for information on how messages are converted to tokens.

# 2、ConversationSummaryMemory的使用

举例1：

如果实例化ConversationSummaryMemory前，没有历史消息，可以使用构造方法实例化

In [6]:


llm = ChatOpenAI(model="deepseek-chat")

memory = ConversationSummaryMemory(llm=llm)

memory.save_context({"input":"你好"},{"output":"怎么了"})
memory.save_context({"input":"你是谁"},{"output":"我是AI助手小智"})
memory.save_context({"input":"初次对话，你能介绍一下你自己吗？"},{"output":"当然可以了。我是一个无所不能的小智。"})

print(memory.load_memory_variables({}))

C:\Users\Song\AppData\Local\Temp\ipykernel_31464\305022390.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm)


{'history': 'The human greets the AI. The AI responds by asking what\'s up. The human then asks "Who are you?" in Chinese, and the AI responds in Chinese, identifying itself as the AI assistant Xiao Zhi. The human then asks Xiao Zhi to introduce itself in this first conversation, and Xiao Zhi replies that it certainly can, describing itself as an all-capable Xiao Zhi.'}


举例2：如果实例化ConversationSummaryMemory前，已经有历史消息，可以调用from_messages()实例化

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory

llm = ChatOpenAI(model="deepseek-chat")

# 假设原始消息

history = ChatMessageHistory()

history.add_user_message("你好，你是谁？")
history.add_ai_message("我是AI助手小智")

# 创建ConversationSummaryMemory的实例
memory = ConversationSummaryMemory.from_messages(
    llm=llm,
    #是生成摘要的原材料，保留完整对话供必要时回溯。当新增对话时，LLM需要结合原始历史生成新摘要
    chat_memory=history
)

print(memory.load_memory_variables({}))

memory.save_context({"input":"我的名字叫小明"},{"output":"很高兴认识你"})

print(memory.load_memory_variables({}))

#记录了历史的交互信息
print(memory.chat_memory.messages)

{'history': 'The human greets the AI and asks who it is. The AI introduces itself as an AI assistant named Xiao Zhi.'}
{'history': 'The human greets the AI and asks who it is. The AI introduces itself as an AI assistant named Xiao Zhi. The human then introduces their own name as Xiao Ming, and the AI responds that it is pleased to meet them.'}


# 3、ConversationSummaryBufferMemory的使用

举例1：

In [14]:
#不支持deepseek


from langchain_classic.memory import ConversationSummaryBufferMemory
llm = ChatOpenAI(model="deepseek-chat")

# 实例化ConversationSummaryBufferMemory的使用
memory = ConversationSummaryBufferMemory(
    llm = llm,
    max_token_limit=40, #控制缓冲区的大小
    return_messages=True
)

#向Memory中存储信息
memory.save_context({"input":"你好，我的名字叫小明"},{"output":"很高兴认识你"})
memory.save_context({"input":"李白是那个朝代的诗人"},{"output":"李白是唐朝的"})
memory.save_context({"input":"唐宋八大家里有苏轼吗"},{"output":"有"})

print(memory.load_memory_variables())

NotImplementedError: get_num_tokens_from_messages() is not presently implemented for model deepseek-chat. See https://platform.openai.com/docs/guides/text-generation/managing-tokens for information on how messages are converted to tokens.

举例2：AI客服场景

In [ ]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1、初始化大语言模型
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    max_tokens=500
)
# 2、定义提示模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是电商客服助手，用中文友好回复用户问题。保持专业但亲切的语气。"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# 3、创建带摘要缓冲的记忆系统
memory = ConversationSummaryBufferMemory(
    llm=llm,
    max_token_limit=400,
    memory_key="chat_history",
    return_messages=True
)
# 4、创建对话链
chain = LLMChain(
    llm=llm,
    prompt=prompt,
    memory=memory,
)
# 5、模拟多轮对话
dialogue = [
("你好，我想查询订单12345的状态", None),
("这个订单是上周五下的", None),
("我现在急着用，能加急处理吗", None),
("等等，我可能记错订单号了，应该是12346", None),
("对了，你们退货政策是怎样的", None)
]
# 6、执行对话
for user_input, _ in dialogue:
    response = chain.invoke({"input": user_input})
    print(f"用户: {user_input}")
    print(f"客服: {response['text']}\n")
# 7、查看当前记忆状态
print("\n=== 当前记忆内容 ===")
print(memory.load_memory_variables({}))

# 4、ConversationEntityMemory的使用（了解）

In [17]:
from langchain_classic.memory import ConversationEntityMemory
from langchain_classic.chains.llm import LLMChain

from langchain_classic.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE

# 初始化大语言模型
llm = ChatOpenAI(model='deepseek-chat', temperature=0)
# 使用LangChain为实体记忆设计的预定义模板
prompt = ENTITY_MEMORY_CONVERSATION_TEMPLATE
# 初始化实体记忆
memory = ConversationEntityMemory(llm=llm)
# 提供对话链
chain = LLMChain(
llm=llm,
prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
memory=ConversationEntityMemory(llm=llm),
#verbose=True, # 设置为True可以看到链的详细推理过程
)
# 进行几轮对话，记忆组件会在后台自动提取和存储实体信息
chain.invoke(input="你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。")
chain.invoke(input="我住在纽约。")
chain.invoke(input="我使用的装备是由斯塔克工业提供的。")
# 查询记忆体中存储的实体信息
print("\n当前存储的实体信息:")
print(chain.memory.entity_store.store)


C:\Users\Song\AppData\Local\Temp\ipykernel_31464\355044348.py:13: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(



当前存储的实体信息:
{'蜘蛛侠': '蜘蛛侠的好朋友包括钢铁侠、美国队长和绿巨人。', '钢铁侠': '钢铁侠是蜘蛛侠的好朋友。', '美国队长': '美国队长是蜘蛛侠的好朋友之一。', '绿巨人': '绿巨人是蜘蛛侠的好朋友之一。', '纽约': '蜘蛛侠住在纽约。', '斯塔克工业': '斯塔克工业为蜘蛛侠提供英雄装备。'}


In [18]:
# 基于记忆进行提问
answer = chain.invoke(input="你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？")
print("\nAI的回答:")
print(answer)


AI的回答:
{'input': '你能告诉我蜘蛛侠住在哪里以及他的好朋友有哪些吗？', 'history': 'Human: 你好，我叫蜘蛛侠。我的好朋友包括钢铁侠、美国队长和绿巨人。\nAI: 你好，蜘蛛侠！很高兴认识你。你和钢铁侠、美国队长、绿巨人组成的团队真是超级英雄界的传奇组合呢。你们一起保护世界、并肩作战的故事总是让人印象深刻。今天有什么想聊聊的吗？\nHuman: 我住在纽约。\nAI: 是的，纽约是你们活动的重要舞台！从皇后区到曼哈顿，复仇者大厦（或者斯塔克大厦）也矗立在那里，这座城市见证了太多你们战斗、成长和守护的时刻。纽约市民虽然时常需要疏散，但一定也为有你们这样的邻居感到安心。你最喜欢纽约的哪个地方？是俯瞰城市的屋顶，还是那家你们常去的披萨店？\nHuman: 我使用的装备是由斯塔克工业提供的。\nAI: 确实，斯塔克工业的科技支持对你来说至关重要！从战衣的智能系统到蛛丝发射器的不断升级，托尼·斯塔克和他的公司为你提供了许多强大而创新的装备。这些技术不仅增强了你的战斗力，也常常在关键时刻帮助你们团队应对各种危机。斯塔克工业的“黑科技”可以说是你英雄生涯中不可或缺的一部分呢。需要聊聊某件特定装备的功能，或者你和托尼在技术上的合作故事吗？', 'entities': {'蜘蛛侠': '蜘蛛侠的好朋友包括钢铁侠、美国队长和绿巨人。'}, 'text': '根据我们的对话，蜘蛛侠（也就是你）住在纽约。而你的好朋友包括钢铁侠、美国队长和绿巨人。需要我补充更多关于你们在纽约的故事或者这些友谊的细节吗？ 😊'}


# 5、ConversationKGMemory的使用（了解）

In [19]:
from langchain_community.memory.kg import ConversationKGMemory

#1.导入相关包
# 2.定义LLM
llm = ChatOpenAI(model="deepseek-chat", temperature=0)
# 3.定义ConversationKGMemory对象
memory = ConversationKGMemory(llm=llm)
# 4.保存会话
memory.save_context({"input": "向山姆问好"}, {"output": "山姆是谁"})
memory.save_context({"input": "山姆是我的朋友"}, {"output": "好的"})
# 5.查询会话
memory.load_memory_variables({"input": "山姆是谁"})

ImportError: Could not import networkx python package. Please install it with `pip install networkx`.

In [ ]:
memory.get_knowledge_triplets("她最喜欢的颜色是红色")

# 6、VectorStoreRetrieverMemory（了解）

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_classic.memory import ConversationBufferMemory, VectorStoreRetrieverMemory
###不支持DeepSeek

import os
import dotenv
from langchain_openai import OpenAIEmbeddings
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("DEEPSEEK_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("DEEPSEEK_BASE_URL")
embeddings_model = OpenAIEmbeddings(
model="text-embedding-ada-002"
)

# 1.导入相关包
# 2.定义ConversationBufferMemory对象
memory = ConversationBufferMemory()
memory.save_context({"input": "我最喜欢的食物是披萨"}, {"output": "很高兴知道"})
memory.save_context({"Human": "我喜欢的运动是跑步"}, {"AI": "好的,我知道了"})
memory.save_context({"Human": "我最喜欢的运动是足球"}, {"AI": "好的,我知道了"})
# 3.定义向量嵌入模型
embeddings_model = OpenAIEmbeddings(
model="text-embedding-ada-002"
)
# 4.初始化向量数据库
vectorstore = FAISS.from_texts(memory.buffer.split("\n"), embeddings_model) # 空初始化
# 5.定义检索对象
retriever = vectorstore.as_retriever(search_kwargs=dict(k=1))
# 6.初始化VectorStoreRetrieverMemory
memory = VectorStoreRetrieverMemory(retriever=retriever)
print(memory.load_memory_variables({"prompt": "我最喜欢的食物是"}))